# Spatial Bayesian search — exploration notebook

This notebook ties the `gemini_proyect` simulator and modeling code together.
It mirrors the four tasks of Deliverable 2:

1. Build a prior over the grid using the accident information.
2. Define a detection model `rho_{t,j}`.
3. Update the prior using simulated mission outcomes.
4. Propose the next mission and compare strategies.

Run each section top-to-bottom. The local simulator is a drop-in replica
of the search-missions webapp, so the same code can be pointed at the
real submission with no changes (only the `SearchEnvironment.run_mission`
call is replaced).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from gemini_proyect.simulator import SearchEnvironment, TrueDetector, load_grid
from gemini_proyect.modeling import (
    DetectionModel,
    drift_prior, drift_prior_with_witnesses, sinking_adjusted_prior, uniform_prior,
    posterior_update,
)
from gemini_proyect.strategies import (
    propose_info_gain, propose_max_expected_detection, propose_max_posterior_rect,
)

GRID_CSV = Path('../data/grid_dataset.csv').resolve()
grid = load_grid(GRID_CSV)
print('grid:', grid.Nx, 'x', grid.Ny, '=', grid.n_cells, 'cells')

## 1. Prior construction

Four candidate priors:

* `uniform` — baseline, no information used.
* `drift` — anisotropic Gaussian centred on the expected landing point,
  using only the physical vectors (plane velocity, wind, water drift).
* `drift+witnesses` — same, with small shifts implied by the two
  witness statements (forward + lateral).
* `drift+witnesses+deep` — multiplied by `exp(depth_bias * depth)` to
  encode a soft prior that the object sinks toward deeper cells.

In [ ]:
priors = {
    'uniform':                uniform_prior(grid),
    'drift':                  drift_prior(grid),
    'drift+witnesses':        drift_prior_with_witnesses(grid),
    'drift+witnesses+deep':   sinking_adjusted_prior(drift_prior_with_witnesses(grid), grid, depth_bias=1.0),
}

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), constrained_layout=True)
for ax, (name, p) in zip(axes, priors.items()):
    im = ax.imshow(grid.reshape_2d(p), origin='lower', cmap='viridis', aspect='auto',
                   extent=[0, grid.Nx, 0, grid.Ny])
    ax.scatter([7], [20], color='red', s=60, label='accident')
    ax.set_title(name)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    fig.colorbar(im, ax=ax, shrink=0.85)
axes[0].legend(loc='upper left')
plt.show()

## 2. Detection model

$$
\rho(j, e) = 1 - \exp\!\bigl(-\lambda_0 \, (1-d_j)^{a_d} (1-r_j)^{a_r} \, e\bigr).
$$

* Always in $[0, 1)$, increasing in effort, decreasing in depth and roughness.
* Effort acts as the number of independent sensor passes (exponential-CDF form).
* Matches the qualitative pattern of all four previous mission reports.

In [ ]:
model = DetectionModel(lambda_0=1.2, a_d=1.0, a_r=0.6)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for ax, e in zip(axes, [1, 2, 3]):
    rho = model.rho(grid.depth, grid.roughness, e)
    im = ax.imshow(grid.reshape_2d(rho), origin='lower', cmap='magma',
                   vmin=0, vmax=1, extent=[0, grid.Nx, 0, grid.Ny], aspect='auto')
    ax.set_title(f'rho with effort = {e}')
    fig.colorbar(im, ax=ax, shrink=0.85)
plt.show()

## 3. Simulated campaign and posterior

Plant a hidden object using a (different) true generating process and run
an information-gain-driven sequence of missions.
The simulator uses `TrueDetector` whose parameters differ from `DetectionModel`,
so we observe how robust the posterior is to misspecification.

In [ ]:
env = SearchEnvironment.from_csv(GRID_CSV, seed=7, detector=TrueDetector())
truth_prior = drift_prior_with_witnesses(grid)
true_cell = env.plant_object(prior=truth_prior)
print('hidden cell:', true_cell, 'xy =', grid.x[true_cell], grid.y[true_cell])

prior = drift_prior_with_witnesses(grid)
posterior = prior.copy()

MAX_MISSIONS = 60
found = False
for _ in range(MAX_MISSIONS):
    if env.budget_remaining < 4: break
    try:
        prop = propose_info_gain(posterior, grid, model, env.budget_remaining)
    except RuntimeError:
        break
    rec = env.run_mission(**prop.as_kwargs())
    posterior = posterior_update(prior, grid, model, env.history)
    if rec.s_t == 1:
        found = True; break

print('found:', found, '| missions:', len(env.history), '| budget used:', env.budget_used)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, arr, name in zip(axes, [prior, posterior], ['prior', 'posterior']):
    im = ax.imshow(grid.reshape_2d(arr), origin='lower', cmap='viridis',
                   extent=[0, grid.Nx, 0, grid.Ny], aspect='auto')
    ax.scatter([grid.x[true_cell]], [grid.y[true_cell]], color='red', s=80,
               marker='x', label='true cell')
    for m in env.history:
        ax.add_patch(plt.Rectangle((m.x_min-0.5, m.y_min-0.5),
                                    m.x_max - m.x_min + 1, m.y_max - m.y_min + 1,
                                    fill=False,
                                    edgecolor='red' if m.s_t else 'black',
                                    alpha=0.5, linewidth=1))
    ax.set_title(name); ax.set_xlabel('x'); ax.set_ylabel('y')
    fig.colorbar(im, ax=ax, shrink=0.85)
axes[0].legend(loc='upper left')
plt.show()

## 4. Propose the next mission from the posterior

Three strategies for picking the rectangle for the next mission:
* `max_expected_detection` — argmax of $\sum_j \pi_j q_{tj} / \text{cost}$.
* `info_gain` — argmax of expected entropy reduction per unit cost.
* `max_posterior_rect` — argmax of $\sum_j \pi_j q_{tj}$ ignoring cost (likes big rectangles).

In [ ]:
for name, fn in [('max_expected_detection', propose_max_expected_detection),
                 ('info_gain',              propose_info_gain),
                 ('max_posterior_rect',     propose_max_posterior_rect)]:
    prop = fn(posterior, grid, model, env.budget_remaining)
    print(f'{name:>22s}: x=[{prop.x_min:.1f},{prop.x_max:.1f}] y=[{prop.y_min:.1f},{prop.y_max:.1f}] '
          f'e={prop.effort} cost={prop.cost:.0f} score={prop.score:.4f}')

## 5. Strategy / prior comparison (Monte Carlo)

For each (strategy, prior) we run ``n_trials`` independent campaigns.
The truth is always drawn from the *witness-informed* drift prior so the
ranking reflects modeling quality, not whether the inference prior happens
to match the truth-generating prior.

In [ ]:
from gemini_proyect.experiments.compare_strategies import run_grid, summarize
df = run_grid(
    strategies=['info_gain', 'max_expected_detection', 'max_posterior_rect'],
    priors=['uniform', 'drift', 'drift+witnesses'],
    n_trials=20,
)
summary = summarize(df)
summary

## 6. Connecting to the real webapp

Once a strategy has been validated locally, switching to the real submission
is mechanical:

* keep `prior`, `model`, `posterior_update`, and the strategy function as-is;
* replace `env.run_mission(...)` with manual entry of the same rectangle and
  effort into the Streamlit form, then read back `s_t` and append to a
  history list that has the same `MissionRecord` shape;
* call `posterior_update(prior, grid, model, history)` to obtain the
  updated posterior, and feed it back into the strategy.